# Análisis y limpieza de datos históricos - PCI
## Limpieza de columnas innecesarias, filtro desde 2005 y consolidación diaria
Este notebook realiza la limpieza del dataset histórico de PCI:
- elimina columnas innecesarias
- filtra registros anteriores a 2005
- agrupa registros por carretera y fecha (año, mes, día)
- usa la mediana de PCI_Score para consolidar registros duplicados en el mismo día

In [11]:
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

data_path = Path(r'../CSVS/PCI-histo.csv')
output_path = Path(r'../CSVS/PCI-histo-cleaned-2005.csv')

print('Ruta de datos:', data_path.resolve())
print('Ruta de salida:', output_path.resolve())

Ruta de datos: C:\Users\Iván\Documents\FP\clase\Proyecto\proyecto-carreteras-sf-fantastic-four\PCI_Historico\CSVS\PCI-histo.csv
Ruta de salida: C:\Users\Iván\Documents\FP\clase\Proyecto\proyecto-carreteras-sf-fantastic-four\PCI_Historico\CSVS\PCI-histo-cleaned-2005.csv


In [12]:
df = pd.read_csv(data_path)
print('Forma original:', df.shape)
print('Columnas originales:', df.columns.tolist())
print(df.head(5).to_string(index=False))

Forma original: (307133, 15)
Columnas originales: ['CNN', 'Street_Name', 'From_Street', 'To_Street', 'PCI_Score', 'PCI_Change_Date', 'Treatment_or_Survey', 'Street_Accepted_For_Maintenance', 'Functional_Class', 'X', 'Y', 'Latitude', 'Longitude', 'Line', 'Point']
     CNN Street_Name   From_Street    To_Street  PCI_Score         PCI_Change_Date Treatment_or_Survey Street_Accepted_For_Maintenance Functional_Class            X            Y  Latitude   Longitude                                                                                                                 Line Point
11007000     Rice St  San Jose Ave   De Long St         80 2012 Mar 13 01:35:18 PM              Survey                             Yes      Residential 5.995097e+06 2.086669e+06 37.709315 -122.458715 LINESTRING (-122.458241194555 37.708999302049, -122.459077441323 37.709557118494, -122.459125957939 37.709665941703)   NaN
11760000  Seneca Ave    Bannock St   Cayuga Ave         62 2016 Jun 07 07:45:15 AM         

In [13]:
# Convertir la fecha a datetime y extraer componentes de fecha
df['PCI_Change_Date'] = pd.to_datetime(df['PCI_Change_Date'], errors='coerce')
df['Year'] = df['PCI_Change_Date'].dt.year
df['Month'] = df['PCI_Change_Date'].dt.month
df['Day'] = df['PCI_Change_Date'].dt.day

print('Fechas inválidas:', df['PCI_Change_Date'].isna().sum())
print('Año mínimo:', df['Year'].min())
print('Año máximo:', df['Year'].max())

Fechas inválidas: 70
Año mínimo: 1947.0
Año máximo: 2026.0


In [ ]:
# Columnas útiles para el análisis
keep_columns = [
    'CNN',
    'Street_Name',
    'PCI_Score',
    'PCI_Change_Date',
    'Treatment_or_Survey',
    'Functional_Class',
    'Latitude',
    'Longitude',
    'Year',
    'Month',
    'Day'
]

df_clean = df[keep_columns].copy()
df_clean['Street_Name'] = df_clean['Street_Name'].fillna('').astype(str).str.lower()
print('Columnas mantenidas:', df_clean.columns.tolist())

Columnas mantenidas: ['CNN', 'Street_Name', 'PCI_Score', 'PCI_Change_Date', 'Treatment_or_Survey', 'Functional_Class', 'Latitude', 'Longitude', 'Year', 'Month', 'Day']


## Clasificar nulos de Functional_Class

Para los registros que tienen `Functional_Class` nulo:
- si `Street_Name` indica una intersección, se marca como `Intersection`
- para los demás se marca como `Other`

In [15]:
intersection_pattern = r'(?i)(?:intersection|:|/|&|\bAT\b|@)'
mask_null_fc = df_clean['Functional_Class'].isna()
mask_intersection = mask_null_fc & df_clean['Street_Name'].str.contains(intersection_pattern, na=False)
mask_other = mask_null_fc & ~mask_intersection

print('Functional_Class nulo antes:', mask_null_fc.sum())
print(' - Intersection:', mask_intersection.sum())
print(' - Other:', mask_other.sum())

df_clean.loc[mask_intersection, 'Functional_Class'] = 'Intersection'
df_clean.loc[mask_other, 'Functional_Class'] = 'Other'

Functional_Class nulo antes: 14117
 - Intersection: 5078
 - Other: 9039


In [16]:
# Calcular PCI anterior por carretera
# Ordenar por carretera y fecha
df_clean = df_clean.sort_values(['CNN', 'PCI_Change_Date']).reset_index(drop=True)

# Crear columna Previous_PCI: para cada carretera, el valor anterior de PCI
df_clean['Previous_PCI'] = df_clean.groupby('CNN')['PCI_Score'].shift(1)

# Calcular el cambio en PCI
df_clean['PCI_Change'] = df_clean['PCI_Score'] - df_clean['Previous_PCI']

print('Información de PCI anterior agregada')
print('\nPrimeros 15 registros por carretera (ordenados por fecha):')
print(df_clean[['CNN', 'Street_Name', 'Year', 'PCI_Score', 'Previous_PCI', 'PCI_Change']].head(15).to_string(index=False))


Información de PCI anterior agregada

Primeros 15 registros por carretera (ordenados por fecha):
   CNN Street_Name   Year  PCI_Score  Previous_PCI  PCI_Change
100000     01st st 1992.0         72           NaN         NaN
100000     01st st 1993.0         72          72.0         0.0
100000     01st st 1994.0         72          72.0         0.0
100000     01st st 1995.0         46          72.0       -26.0
100000     01st st 1996.0         62          46.0        16.0
100000     01st st 1997.0         51          62.0       -11.0
100000     01st st 2001.0        100          51.0        49.0
100000     01st st 2002.0        100         100.0         0.0
100000     01st st 2005.0        100         100.0         0.0
100000     01st st 2007.0        100         100.0         0.0
100000     01st st 2009.0         93         100.0        -7.0
100000     01st st 2010.0         73          93.0       -20.0
100000     01st st 2013.0         56          73.0       -17.0
100000     01st st 20

In [17]:
# Filtrar registros desde 2005 en adelante
df_clean = df_clean[df_clean['Year'] >= 2005].copy()

# Recalcular PCI anterior a partir del conjunto filtrado,
# y ajustar los primeros registros de cada carretera que ya no tienen historial previo.
# En esos casos, asumimos Previous_PCI = PCI_Score y PCI_Change = 0.
df_clean = df_clean.sort_values(['CNN', 'PCI_Change_Date']).reset_index(drop=True)
df_clean['Previous_PCI'] = df_clean.groupby('CNN')['PCI_Score'].shift(1)
df_clean['PCI_Change'] = df_clean['PCI_Score'] - df_clean['Previous_PCI']
mask_no_prev = df_clean['Previous_PCI'].isna()
df_clean.loc[mask_no_prev, 'Previous_PCI'] = df_clean.loc[mask_no_prev, 'PCI_Score']
df_clean.loc[mask_no_prev, 'PCI_Change'] = 0

print('Registros después de filtrar desde 2005:', df_clean.shape)

Registros después de filtrar desde 2005: (210020, 13)


In [18]:
# Consolidar duplicados por carretera y fecha
print('Duplicados antes de agrupar:', df_clean.duplicated(subset=['CNN', 'Year', 'Month', 'Day']).sum())
df_grouped = df_clean.groupby(['CNN', 'Year', 'Month', 'Day'], as_index=False).agg({
    'PCI_Score': 'median',
    'Previous_PCI': 'first',
    'PCI_Change': 'first',
    'Street_Name': 'first',
    'PCI_Change_Date': 'first',
    'Treatment_or_Survey': 'first',
    'Functional_Class': 'first',
    'Latitude': 'median',
    'Longitude': 'median'
})
print('Forma después de agrupar:', df_grouped.shape)
print('Duplicados después de agrupar:', df_grouped.duplicated(subset=['CNN', 'Year', 'Month', 'Day']).sum())
print('\nPrimeras filas con PCI anterior:')
print(df_grouped[['CNN', 'Street_Name', 'Year', 'PCI_Score', 'Previous_PCI', 'PCI_Change']].head(10).to_string(index=False))

Duplicados antes de agrupar: 6752
Forma después de agrupar: (203268, 13)
Duplicados después de agrupar: 0

Primeras filas con PCI anterior:
   CNN Street_Name   Year  PCI_Score  Previous_PCI  PCI_Change
100000     01st st 2005.0      100.0         100.0         0.0
100000     01st st 2007.0      100.0         100.0         0.0
100000     01st st 2009.0       93.0         100.0        -7.0
100000     01st st 2010.0       73.0          93.0       -20.0
100000     01st st 2013.0       56.0          73.0       -17.0
100000     01st st 2014.0       61.0          56.0         5.0
100000     01st st 2015.0       65.0          61.0         4.0
100000     01st st 2015.0       69.0          65.0         4.0
100000     01st st 2017.0       80.0          69.0        11.0
100000     01st st 2018.0       79.0          80.0        -1.0


In [19]:
df_final = df_grouped.drop(columns=['Year', 'Month', 'Day'])

In [20]:
# Guardar dataframe limpio
df_final.to_csv(output_path, index=False)
print('Dataset limpio guardado en:', output_path.resolve())

Dataset limpio guardado en: C:\Users\Iván\Documents\FP\clase\Proyecto\proyecto-carreteras-sf-fantastic-four\PCI_Historico\CSVS\PCI-histo-cleaned-2005.csv
